In [10]:
"""
═══════════════════════════════════════════════════════
Central configuration, all typed data structures, uncertainty enums,
retrieval decision types, and terminal logger.

Adaptive RAG Novelty:
  Every previous RAG system retrieves unconditionally — given a query, it
  always fetches passages. This ignores a fundamental question: does this
  query even *need* retrieval?

  Adaptive RAG answers this question by measuring the LLM's uncertainty
  before deciding what to do:

    Uncertainty → low   : answer directly (no retrieval needed)
    Uncertainty → medium: retrieve shallowly (1 round, k=3)
    Uncertainty → high  : retrieve deeply (3 rounds, k=10, verify)
    Conflict detected   : escalate (hedge answer, flag for review)
    Query ambiguous     : clarify before retrieving
    Out of knowledge    : route to external search or refuse

  Uncertainty is measured via three independent signals:
    1. Token entropy   — entropy of the LLM's token probability distribution
    2. Sample variance — semantic variance across N sampled completions
    3. Self-assessment — LLM's direct confidence estimate (0–1)

  These are fused into a calibrated UncertaintyScore that drives all
  subsequent decisions in the system.

References:
  • Adaptive RAG (Jeong et al. 2024)          https://arxiv.org/abs/2403.14403
  • Self-RAG (Asai et al. 2023)               https://arxiv.org/abs/2310.11511
  • FLARE (Jiang et al. 2023)                 https://arxiv.org/abs/2305.06983
  • Calibration of LLMs (Kadavath et al.)     https://arxiv.org/abs/2207.05221
  • Semantic Uncertainty (Kuhn et al. 2023)   https://arxiv.org/abs/2302.09664
  • Selective Prediction (Varshney et al.)    https://arxiv.org/abs/2205.09598
  • DRAGIN (Su et al. 2024)                   https://arxiv.org/abs/2403.10081
  • IRCoT (Trivedi et al. 2022)               https://arxiv.org/abs/2212.10509
  • Confident Adaptive LM (Schuster 2022)     https://arxiv.org/abs/2207.07061
  • UAG (He et al. 2024)                      https://arxiv.org/abs/2401.06567
"""

"\n═══════════════════════════════════════════════════════\nCentral configuration, all typed data structures, uncertainty enums,\nretrieval decision types, and terminal logger.\n\nAdaptive RAG Novelty:\n  Every previous RAG system retrieves unconditionally — given a query, it\n  always fetches passages. This ignores a fundamental question: does this\n  query even *need* retrieval?\n\n  Adaptive RAG answers this question by measuring the LLM's uncertainty\n  before deciding what to do:\n\n    Uncertainty → low   : answer directly (no retrieval needed)\n    Uncertainty → medium: retrieve shallowly (1 round, k=3)\n    Uncertainty → high  : retrieve deeply (3 rounds, k=10, verify)\n    Conflict detected   : escalate (hedge answer, flag for review)\n    Query ambiguous     : clarify before retrieving\n    Out of knowledge    : route to external search or refuse\n\n  Uncertainty is measured via three independent signals:\n    1. Token entropy   — entropy of the LLM's token probability distri

In [11]:
from __future__ import annotations

import os
import time
from enum import Enum
from typing import Any, Dict, List, Optional, Tuple
from dataclasses import dataclass, field


In [38]:


# ══════════════════════════════════════════════════════════════════════
# CONFIGURATION
# ══════════════════════════════════════════════════════════════════════

@dataclass
class Config:
    # ── Azure OpenAI (AI Foundry) ─────────────────────────────────────
    AI_FOUNDRY_PROJECT_ENDPOINT: str = "https://idkopenai.services.ai.azure.com"
    AI_FOUNDRY_DEPLOYMENT_NAME: str = "gpt-4o"
    AI_FOUNDRY_API_VERSION: str = "2024-12-01-preview"
    AI_FOUNDRY_API_KEY: str = ""

    # ── HuggingFace (local embeddings) ───────────────────────────────
    EMBEDDING_MODEL: str  = "sentence-transformers/all-MiniLM-L6-v2"
    EMBEDDING_DEVICE: str = field(default_factory=lambda: os.getenv("EMBEDDING_DEVICE", "cpu"))

    # ── Storage ───────────────────────────────────────────────────────
    BASE_DIR: str          = "./adaptive_rag_store"
    CHROMA_DIR: str        = "./adaptive_rag_store/chroma"
    COLLECTION_NAME: str   = "adaptive_rag_kb"

    # ── Chunking ──────────────────────────────────────────────────────
    CHUNK_SIZE: int        = 600
    CHUNK_OVERLAP: int     = 100

    # ── Uncertainty Thresholds ────────────────────────────────────────
    # Three-tier uncertainty bands (Jeong et al. 2024 framework)
    CERTAINTY_THRESHOLD: float   = 0.75  # score ≥ this → SKIP retrieval
    AMBIGUITY_THRESHOLD: float   = 0.30  # score ≤ this → CLARIFY or MULTI_HOP
    # Between 0.30 and 0.75 → standard RETRIEVE

    # ── Uncertainty Estimation ────────────────────────────────────────
    N_SAMPLES: int         = 3    # completions sampled for variance estimation
    SAMPLE_TEMPERATURE: float = 0.7   # temperature for sample generation
    ENTROPY_MAX: float     = 4.0      # normalisation constant for token entropy

    # ── Retrieval Depth (uncertainty-driven) ─────────────────────────
    # SHALLOW: fast, single-shot, low k
    SHALLOW_K: int         = 3
    SHALLOW_ROUNDS: int    = 1
    # STANDARD: default retrieval
    STANDARD_K: int        = 6
    STANDARD_ROUNDS: int   = 2
    # DEEP: high-uncertainty, multi-hop, iterative verification
    DEEP_K: int            = 10
    DEEP_ROUNDS: int       = 3

    # ── Retrieval params ──────────────────────────────────────────────
    FETCH_K: int           = 16
    MMR_LAMBDA: float      = 0.65

    # ── Conflict detection ────────────────────────────────────────────
    CONFLICT_SIM_THRESHOLD: float = 0.30  # semantic similarity below this → conflict
    CONFLICT_COSINE_FLIP: float   = -0.10  # cosine below this → directional conflict

    # ── Knowledge boundary ────────────────────────────────────────────
    # Queries with avg embedding distance > this to KB → likely out-of-scope
    OOK_DISTANCE_THRESHOLD: float = 0.80   # out-of-knowledge threshold (cosine dist)

    # ── Answer calibration ────────────────────────────────────────────
    HIGH_CONF_THRESHOLD: float    = 0.80
    MEDIUM_CONF_THRESHOLD: float  = 0.50

    # ── Retrieval necessity predictor ─────────────────────────────────
    # Heuristic features that signal NO retrieval needed
    FACTUAL_QUESTION_PATTERNS: Tuple[str, ...] = (
        "what is", "define ", "who is", "when was", "where is",
        "how many", "calculate", "convert",
    )
    OPINION_PATTERNS: Tuple[str, ...] = (
        "what do you think", "your opinion", "do you believe", "should i",
    )

    # ── LLM ──────────────────────────────────────────────────────────
    LLM_TEMPERATURE: float  = 0.0
    LLM_MAX_TOKENS: int     = 1400

    def is_configured(self) -> bool:
        """Check Azure OpenAI credentials are configured."""
        return bool(
            self.AI_FOUNDRY_PROJECT_ENDPOINT
            and self.AI_FOUNDRY_DEPLOYMENT_NAME
            and self.AI_FOUNDRY_API_VERSION
            and self.AI_FOUNDRY_API_KEY
        )


In [43]:

# ══════════════════════════════════════════════════════════════════════
# CORE ENUMS
# ══════════════════════════════════════════════════════════════════════

class UncertaintyLevel(str, Enum):
    CERTAIN    = "certain"    # score ≥ 0.75 — LLM knows the answer
    MODERATE   = "moderate"   # score 0.30–0.75 — retrieval helpful
    UNCERTAIN  = "uncertain"  # score < 0.30 — retrieval essential
    CONFLICTED = "conflicted" # parametric belief ≠ retrieved evidence


class RetrievalDecision(str, Enum):
    SKIP       = "skip"       # answer from parametric knowledge, no retrieval
    RETRIEVE   = "retrieve"   # standard retrieval (depth depends on uncertainty)
    CLARIFY    = "clarify"    # query is ambiguous — ask for disambiguation
    MULTI_HOP  = "multi_hop"  # complex multi-step query — chained retrievals
    REFUSE     = "refuse"     # out-of-knowledge, potentially harmful


class RetrievalDepth(str, Enum):
    SHALLOW  = "shallow"   # 1 round, k=3
    STANDARD = "standard"  # 2 rounds, k=6
    DEEP     = "deep"      # 3 rounds, k=10


class ConflictType(str, Enum):
    NONE        = "none"        # no conflict detected
    MINOR       = "minor"       # small factual discrepancy
    MAJOR       = "major"       # direct contradiction
    TEMPORAL    = "temporal"    # parametric knowledge is outdated
    DIRECTIONAL = "directional" # opposing claims (cosine flip)


class AnswerConfidence(str, Enum):
    HIGH    = "high"    # ≥ 0.80
    MEDIUM  = "medium"  # 0.50–0.80
    LOW     = "low"     # < 0.50
    HEDGED  = "hedged"  # conflict detected — answer hedged


In [14]:


# ══════════════════════════════════════════════════════════════════════
# UNCERTAINTY SCORE
# ══════════════════════════════════════════════════════════════════════

@dataclass
class UncertaintyScore:
    """
    Calibrated uncertainty score fused from three independent signals.

    Based on Semantic Uncertainty (Kuhn et al., 2023) and
    Calibration of LLMs (Kadavath et al., 2022).

    Fusion:
      score = w_e·(1 − norm_entropy) + w_v·(1 − sample_variance) + w_s·self_score
      Default weights: w_e=0.30, w_v=0.40, w_s=0.30
    """
    # Raw signals
    token_entropy:     float = 0.5   # H(token dist) / H_max, 0=certain, 1=uniform
    sample_variance:   float = 0.5   # semantic variance across N samples, 0=consistent
    self_score:        float = 0.5   # LLM's direct confidence estimate, 0–1

    # Fusion weights
    w_entropy:   float = 0.30
    w_variance:  float = 0.40
    w_self:      float = 0.30

    # Metadata
    query:       str = ""
    signal_notes: Dict[str, Any] = field(default_factory=dict)

    @property
    def fused_score(self) -> float:
        """
        Higher = more certain. Range [0, 1].
          entropy  : lower entropy → more certain
          variance : lower variance → more certain
          self     : higher self_score → more certain
        """
        certain_from_entropy  = 1.0 - min(1.0, self.token_entropy)
        certain_from_variance = 1.0 - min(1.0, self.sample_variance)
        certain_from_self     = max(0.0, min(1.0, self.self_score))
        return (
            self.w_entropy   * certain_from_entropy
            + self.w_variance  * certain_from_variance
            + self.w_self      * certain_from_self
        )

    @property
    def level(self) -> UncertaintyLevel:
        s = self.fused_score
        if s >= 0.75:
            return UncertaintyLevel.CERTAIN
        if s >= 0.30:
            return UncertaintyLevel.MODERATE
        return UncertaintyLevel.UNCERTAIN

    def describe(self) -> str:
        return (
            f"score={self.fused_score:.3f} [{self.level.value}] "
            f"| entropy={self.token_entropy:.2f} "
            f"| variance={self.sample_variance:.2f} "
            f"| self={self.self_score:.2f}"
        )


# ══════════════════════════════════════════════════════════════════════
# RETRIEVAL DECISION RECORD
# ══════════════════════════════════════════════════════════════════════

@dataclass
class RetrievalDecisionRecord:
    """
    Full record of a retrieval decision: why this decision was made,
    what signals drove it, what depth was chosen.
    """
    query:           str
    decision:        RetrievalDecision
    depth:           Optional[RetrievalDepth]
    uncertainty:     UncertaintyScore
    rationale:       str            # human-readable explanation
    ook_signal:      bool = False   # out-of-knowledge flag
    ambiguity_score: float = 0.0    # 0 = unambiguous, 1 = maximally ambiguous
    is_multi_hop:    bool = False   # complex chained query
    heuristic_skip:  bool = False   # skipped via lightweight heuristic (no LLM call)
    timestamp:       float = field(default_factory=time.time)


# ══════════════════════════════════════════════════════════════════════
# CONFLICT RECORD
# ══════════════════════════════════════════════════════════════════════

@dataclass
class ConflictRecord:
    """
    Records a detected conflict between parametric belief and retrieved evidence.
    Drives the conflict resolution strategy.
    """
    conflict_type:       ConflictType
    parametric_claim:    str       # what the LLM believed before retrieval
    evidence_claim:      str       # what the retrieved evidence says
    conflict_score:      float     # 0 (no conflict) to 1 (full contradiction)
    resolution:          str       # how the conflict was handled
    evidence_source:     str       # source of conflicting evidence


# ══════════════════════════════════════════════════════════════════════
# ADAPTIVE RESULT
# ══════════════════════════════════════════════════════════════════════

@dataclass
class AdaptiveRAGResult:
    """Full result from an Adaptive RAG query."""
    query:               str
    decision_record:     RetrievalDecisionRecord
    uncertainty:         UncertaintyScore
    retrieved_docs:      List[Any]                   # LangChain Documents
    parametric_draft:    str                         # initial LLM draft (before retrieval)
    final_answer:        str                         # final answer (after optional retrieval)
    answer_confidence:   AnswerConfidence
    confidence_score:    float                       # calibrated 0–1
    conflict:            Optional[ConflictRecord]
    retrieval_rounds:    int                         # how many retrieval rounds ran
    clarification_needed: bool                       # True if query needs clarification
    clarification_questions: List[str]               # suggested clarification Qs
    sources:             List[str]
    elapsed_sec:         float = 0.0
    notes:               List[str] = field(default_factory=list)  # audit trail


# ══════════════════════════════════════════════════════════════════════
# MULTI-HOP CHAIN RECORD
# ══════════════════════════════════════════════════════════════════════

@dataclass
class HopRecord:
    """One step in a multi-hop retrieval chain."""
    hop_num:         int
    sub_query:       str
    retrieved:       List[Any]   # docs retrieved this hop
    partial_answer:  str         # answer assembled so far
    uncertainty:     UncertaintyScore  # uncertainty after this hop
    continue_chain:  bool        # should we continue to next hop?


# ══════════════════════════════════════════════════════════════════════
# TERMINAL LOGGER
# ══════════════════════════════════════════════════════════════════════

class C:
    RESET  = "\033[0m"; BOLD  = "\033[1m"; DIM   = "\033[2m"
    CYAN   = "\033[96m"; GREEN= "\033[92m"; YELLOW= "\033[93m"
    RED    = "\033[91m"; MAG  = "\033[95m"; BLUE  = "\033[94m"
    WHITE  = "\033[97m"; ORANGE="\033[33m"; TEAL  = "\033[36m"
    # Semantic colours for uncertainty levels
    CERT   = "\033[92m"   # green   — certain
    MOD    = "\033[93m"   # yellow  — moderate
    UNC    = "\033[91m"   # red     — uncertain
    CONF   = "\033[95m"   # magenta — conflict
    GATE   = "\033[94m"   # blue    — decision gate
    HOP    = "\033[33m"   # orange  — multi-hop
    SKIP   = "\033[96m"   # cyan    — skip (no retrieval)


W = 76

UNCERTAINTY_COLOURS = {
    UncertaintyLevel.CERTAIN:    C.CERT,
    UncertaintyLevel.MODERATE:   C.MOD,
    UncertaintyLevel.UNCERTAIN:  C.UNC,
    UncertaintyLevel.CONFLICTED: C.CONF,
}

DECISION_COLOURS = {
    RetrievalDecision.SKIP:      C.SKIP,
    RetrievalDecision.RETRIEVE:  C.CYAN,
    RetrievalDecision.CLARIFY:   C.YELLOW,
    RetrievalDecision.MULTI_HOP: C.HOP,
    RetrievalDecision.REFUSE:    C.RED,
}


class Log:
    @staticmethod
    def banner(title: str, sub: str = ""):
        print(f"\n{C.BOLD}{C.WHITE}{'═'*W}")
        print(f"  {title}")
        if sub:
            print(f"  {C.DIM}{sub}{C.RESET}{C.BOLD}{C.WHITE}")
        print(f"{'═'*W}{C.RESET}")

    @staticmethod
    def section(title: str, colour: str = C.CYAN):
        print(f"\n{C.BOLD}{colour}{'━'*W}")
        print(f"  {title}")
        print(f"{'━'*W}{C.RESET}")

    @staticmethod
    def step(msg: str):
        print(f"\n{C.BOLD}{C.BLUE}▶ {msg}{C.RESET}")

    @staticmethod
    def ok(msg: str):
        print(f"{C.GREEN}  ✓ {msg}{C.RESET}")

    @staticmethod
    def warn(msg: str):
        print(f"{C.YELLOW}  ⚠  {msg}{C.RESET}")

    @staticmethod
    def err(msg: str):
        print(f"{C.RED}  ✗ {msg}{C.RESET}")

    @staticmethod
    def info(msg: str):
        print(f"{C.DIM}  · {msg}{C.RESET}")

    @staticmethod
    def uncertainty(u: UncertaintyScore):
        col  = UNCERTAINTY_COLOURS.get(u.level, C.WHITE)
        bar  = int(u.fused_score * 20)
        full = "█" * bar + "░" * (20 - bar)
        print(
            f"\n  {C.BOLD}Uncertainty Score{C.RESET}\n"
            f"  {col}{full}{C.RESET} {u.fused_score:.3f} [{u.level.value.upper()}]\n"
            f"  {C.DIM}entropy={u.token_entropy:.2f}  "
            f"variance={u.sample_variance:.2f}  "
            f"self={u.self_score:.2f}{C.RESET}"
        )

    @staticmethod
    def decision(rec: RetrievalDecisionRecord):
        col  = DECISION_COLOURS.get(rec.decision, C.WHITE)
        depth_str = f" → depth={rec.depth.value}" if rec.depth else ""
        print(
            f"\n  {C.BOLD}Retrieval Decision Gate{C.RESET}\n"
            f"  {col}{C.BOLD}▶ {rec.decision.value.upper()}{depth_str}{C.RESET}\n"
            f"  {C.DIM}Rationale: {rec.rationale}{C.RESET}"
        )

    @staticmethod
    def conflict(c: ConflictRecord):
        print(
            f"\n  {C.CONF}{C.BOLD}⚡ CONFLICT DETECTED [{c.conflict_type.value}]{C.RESET}\n"
            f"  {C.DIM}Parametric: {c.parametric_claim[:80]}\n"
            f"  Evidence  : {c.evidence_claim[:80]}\n"
            f"  Score     : {c.conflict_score:.2f}\n"
            f"  Resolution: {c.resolution}{C.RESET}"
        )

    @staticmethod
    def hop(h: HopRecord):
        col = C.HOP
        print(
            f"  {col}[HOP {h.hop_num}]{C.RESET} {h.sub_query[:60]}"
            f"  {C.DIM}docs={len(h.retrieved)} "
            f"uncertainty={h.uncertainty.fused_score:.2f} "
            f"continue={h.continue_chain}{C.RESET}"
        )

    @staticmethod
    def answer_box(result: AdaptiveRAGResult):
        import textwrap
        conf_col = (C.GREEN  if result.answer_confidence == AnswerConfidence.HIGH else
                    C.YELLOW if result.answer_confidence == AnswerConfidence.MEDIUM else
                    C.RED)
        print(f"\n{C.BOLD}{C.GREEN}{'═'*W}")
        print(f"  ADAPTIVE RAG ANSWER  (Uncertainty-Aware Retrieval)")
        print(f"{'═'*W}{C.RESET}")
        print(f"{C.BOLD}  Q: {result.query}{C.RESET}\n")
        for line in textwrap.wrap(result.final_answer, W - 4):
            print(f"  {line}")
        print(f"\n{C.DIM}{'─'*W}")
        print(f"  Decision          : {result.decision_record.decision.value}")
        print(f"  Uncertainty level : {result.uncertainty.level.value} "
              f"(score={result.uncertainty.fused_score:.3f})")
        print(f"  Retrieval rounds  : {result.retrieval_rounds}")
        print(f"  Conflict          : "
              f"{result.conflict.conflict_type.value if result.conflict else 'none'}")
        print(f"  Answer confidence : "
              f"{conf_col}{result.answer_confidence.value} "
              f"({result.confidence_score:.1%}){C.RESET}")
        if result.clarification_needed:
            print(f"  {C.YELLOW}Clarification needed:{C.RESET}")
            for q in result.clarification_questions[:2]:
                print(f"    • {q}")
        if result.sources:
            print(f"  Sources           : {result.sources[:4]}")
        if result.notes:
            print(f"  Audit trail       :")
            for note in result.notes[-4:]:
                print(f"    {C.DIM}• {note}{C.RESET}")
        print(f"  Elapsed           : {result.elapsed_sec:.2f}s")
        print(f"{'─'*W}{C.RESET}")


In [15]:
"""
═══════════════════════════════════════════════════════
Knowledge base corpus: 10 reference papers on uncertainty quantification,
LLM calibration, and adaptive retrieval — all publicly available.

Demo documents: 3 rich technical documents used to test uncertainty-aware
retrieval across different knowledge domains.

Primary PDF auto-download:
  Adaptive RAG (Jeong et al., 2024)  https://arxiv.org/pdf/2403.14403
"""

'\n═══════════════════════════════════════════════════════\nKnowledge base corpus: 10 reference papers on uncertainty quantification,\nLLM calibration, and adaptive retrieval — all publicly available.\n\nDemo documents: 3 rich technical documents used to test uncertainty-aware\nretrieval across different knowledge domains.\n\nPrimary PDF auto-download:\n  Adaptive RAG (Jeong et al., 2024)  https://arxiv.org/pdf/2403.14403\n'

In [44]:

# ══════════════════════════════════════════════════════════════════════
# KNOWLEDGE BASE — 10 reference papers
# ══════════════════════════════════════════════════════════════════════

KB_DOCS: List[Dict[str, str]] = [
    {
        "id": "adaptive_rag_jeong",
        "source": "Adaptive-RAG: Learning to Adapt Retrieval-Augmented LLMs — Jeong et al. (2024)",
        "url": "https://arxiv.org/abs/2403.14403",
        "content": """
Adaptive-RAG trains a small classifier to predict the complexity of incoming queries
and route them to one of three retrieval strategies: no-retrieval (for simple queries
answerable from parametric knowledge), single-step retrieval, or multi-step retrieval.

Query complexity classes:
  A: Simple — answerable from parametric knowledge; retrieval unnecessary or harmful
  B: Single-hop — one retrieval round is sufficient
  C: Multi-hop — requires chained retrieval across multiple passages

The complexity classifier is a lightweight model (T5-small fine-tuned on ~5K examples)
achieving 87.3% accuracy on routing decisions. Using the correct strategy per query:
  No-retrieval accuracy: 71.2% on SimpleQA
  Single-step: 62.4% on NQ Open
  Multi-step: 48.1% on MuSiQue

Key insight: applying retrieval to already-known queries (Class A) decreases accuracy
by 4–8% due to noise from retrieved context. Adaptive routing eliminates this degradation.

Strategy selection cost: the lightweight classifier adds only 12ms overhead per query,
far less than one retrieval round (~200ms) or LLM generation (~500ms).

Adaptive-RAG outperforms always-retrieve baselines by 8.1% on average across 5 QA datasets.
"""
    },
    {
        "id": "self_rag",
        "source": "Self-RAG: Learning to Retrieve, Generate and Critique — Asai et al. (2023)",
        "url": "https://arxiv.org/abs/2310.11511",
        "content": """
Self-RAG trains a single LM to use four types of reflection tokens to self-assess
retrieval necessity and output quality. Reflection tokens are generated inline with
the output text, making retrieval and critique part of the generation process.

Reflection token taxonomy:
  [Retrieve=Yes/No/Continue]  : should retrieval happen now?
  [IsREL=relevant/irrelevant] : is the retrieved passage relevant to the query?
  [IsSUP=supported/partially/unsupported]: is generated output grounded in evidence?
  [IsUSE=1-5]                 : overall utility of the response

Training: fine-tune on output sequences that include reflection tokens at appropriate
positions, using a teacher model to annotate training examples with correct reflections.

Selective retrieval reduces hallucination: for factoid queries, Self-RAG retrieves
and grades evidence; for opinion queries, it skips retrieval entirely.

Results vs always-retrieve baselines:
  PopQA: 54.9% (Self-RAG 7B) vs 50.8% (ChatGPT)
  FactScore biography: 72.4% vs 60.1%
  ARC-Challenge: 67.3% vs 66.2%
  ASQA EM-recall: 38.2 vs 35.1 (standard RAG)

The reflect-then-retrieve paradigm is the conceptual ancestor of Adaptive RAG's
uncertainty-gated decision gate.
"""
    },
    {
        "id": "flare",
        "source": "FLARE: Forward-Looking Active Retrieval — Jiang et al. (2023)",
        "url": "https://arxiv.org/abs/2305.06983",
        "content": """
FLARE monitors token probability during generation. When any token in a tentative
sentence has probability below threshold δ (default 0.4), FLARE treats this as an
uncertainty signal and triggers retrieval for that span.

Algorithm in detail:
  1. Generate next sentence tentatively (full generation)
  2. Check: any token probability < δ?
  3. If yes: use the generated tokens (with low-prob tokens masked) as retrieval query
  4. Retrieve passages, regenerate the sentence with new context
  5. Repeat for each sentence in the output

The connection to Adaptive RAG's uncertainty model: token probability is one component
of FLARE's uncertainty signal. Adaptive RAG generalises this to a three-signal fusion
(token entropy + sample variance + self-assessment).

FLARE benchmarks:
  StrategyQA: 60.1% vs 57.8% (standard RAG)
  ASQA EM-recall: 43.1% vs 39.7%
  2WikiMultihopQA: 46.2% vs 37.9%
  IIRC: 44.6% vs 38.2%

Token-level uncertainty is the finest-grained signal — it fires within a single
sentence generation, enabling extremely precise retrieval triggers.
"""
    },
    {
        "id": "calibration_kadavath",
        "source": "Language Models (Mostly) Know What They Know — Kadavath et al. (2022)",
        "url": "https://arxiv.org/abs/2207.05221",
        "content": """
Large language models can self-assess their own confidence with reasonable accuracy.
When prompted "Is the following statement true? [statement]", models produce well-
calibrated probability estimates — larger models are better calibrated.

Key findings:
  • Claude-scale models calibrate substantially better than smaller models
  • Self-assessment calibration (ECE): 0.08 on TruthfulQA (well-calibrated)
  • Models overestimate confidence on out-of-distribution queries
  • Multi-sample consistency strongly correlates with actual accuracy:
    if 4/5 sampled answers agree, the answer is correct ~85% of the time

P(True) protocol: "Proposed answer: [answer]. Is this answer correct? (yes/no)"
Models can directly output probability of their answer being true.

This paper motivates Adaptive RAG's self_score component of UncertaintyScore.
Specifically: UncertaintyScore.self_score ← LLM.P(my answer is correct | question)

Calibration degrades for:
  - Questions requiring very recent knowledge (post training cutoff)
  - Questions in languages underrepresented in training data
  - Multi-step reasoning questions with long dependency chains
"""
    },
    {
        "id": "semantic_uncertainty",
        "source": "Semantic Uncertainty: Linguistic Invariances for Uncertainty Estimation — Kuhn et al. (2023)",
        "url": "https://arxiv.org/abs/2302.09664",
        "content": """
Standard token-level entropy fails for open-ended generation because multiple different
token sequences can express the same semantic content. Semantic uncertainty clusters
multiple sampled responses by semantic equivalence, then computes entropy over clusters
rather than over token sequences.

Semantic entropy algorithm:
  1. Sample N responses {r_1, ..., r_N} at temperature T=0.7
  2. For each pair (r_i, r_j): compute bidirectional NLI entailment
     → if r_i entails r_j AND r_j entails r_i: same semantic cluster
  3. Compute entropy H = -Σ p(cluster) log p(cluster)
  4. High entropy → high semantic uncertainty → trigger retrieval

Semantic uncertainty outperforms raw token entropy as an uncertainty signal:
  AUROC for detecting wrong answers: semantic entropy 0.79 vs token entropy 0.71
  TriviaQA, NQ, CoQA, SciQ — consistent improvement across all benchmarks

Adaptive RAG's sample_variance component approximates semantic uncertainty using
embedding cosine distance between sampled completions (computationally cheaper than NLI).
"""
    },
    {
        "id": "selective_prediction",
        "source": "Selective Prediction for LLMs — Varshney et al. (2022)",
        "url": "https://arxiv.org/abs/2205.09598",
        "content": """
Selective prediction allows a model to abstain from answering when it is not confident,
rather than hallucinating a wrong answer. This is the "refuse" branch of Adaptive RAG's
decision gate.

Abstention strategies:
  Score-based: abstain if confidence score < threshold
  Set-based:   produce a set of plausible answers; abstain if set is too large
  Cascade:     defer to a stronger model or retrieval system

Coverage-Accuracy trade-off:
  Coverage = fraction of queries answered (not abstained)
  Accuracy = fraction of answered queries that are correct
  Selective prediction improves accuracy at cost of some coverage

Results on NaturalQuestions:
  Always answer: 65.4% accuracy, 100% coverage
  Selective (threshold=0.6): 78.2% accuracy, 71% coverage
  Selective (threshold=0.8): 88.1% accuracy, 52% coverage

In Adaptive RAG: REFUSE decision corresponds to abstention for out-of-knowledge queries.
The system outputs a calibrated uncertainty score with the refusal, allowing the user
or a downstream system to decide whether to seek another source.
"""
    },
    {
        "id": "dragin",
        "source": "DRAGIN: Dynamic Retrieval Augmented Generation based on Information Needs — Su et al. (2024)",
        "url": "https://arxiv.org/abs/2403.10081",
        "content": """
DRAGIN determines when to retrieve (timing) and what to query (content) dynamically
during generation, by tracking the model's real-time information needs.

Two components:
  RIND (Real-time Information Needs Detection): measures token-level uncertainty via
    entropy of the probability distribution over the vocabulary at each generation step.
    Retrieval triggered when H(token) > δ for consecutive tokens.

  RIQG (Real-time Query Generation): identifies which tokens in the generated text
    are the source of uncertainty and constructs the retrieval query from those tokens.
    "Responsible tokens" = tokens with highest individual entropy contribution.

DRAGIN vs baselines on 4-hop reasoning tasks:
  2WikiMultiHopQA: 44.2% (DRAGIN) vs 38.1% (FLARE) vs 35.7% (IRCoT)
  MuSiQue: 31.4% vs 27.2% vs 26.8%
  HotpotQA: 50.1% vs 46.2% vs 44.9%

Adaptive RAG borrows from DRAGIN the idea that retrieval timing should be driven by
real-time uncertainty, and that the retrieval query should focus on the uncertain tokens
rather than the full generation.
"""
    },
    {
        "id": "ircot",
        "source": "IRCoT: Interleaving Retrieval with Chain-of-Thought Reasoning — Trivedi et al. (2022)",
        "url": "https://arxiv.org/abs/2212.10509",
        "content": """
IRCoT interleaves retrieval with chain-of-thought reasoning steps. At each reasoning
step, the model generates a thought, retrieves relevant passages conditioned on that
thought, and incorporates them into the next reasoning step.

Algorithm:
  Initialize: question Q
  Repeat:
    thought_t = LLM.generate_cot_step(Q + context_so_far)
    docs_t    = retrieve(thought_t)  ← query is the latest CoT sentence
    context   = context + thought_t + docs_t
  Until: [Final Answer: ...] token generated or max_steps reached

The multi-hop chain in Adaptive RAG is directly inspired by IRCoT: each hop generates
a sub-query from the current partial answer, retrieves, and checks if the accumulated
evidence is sufficient to answer the original question.

IRCoT results:
  HotpotQA: 56.4 F1 (IRCoT) vs 44.1 (one-shot RAG)
  MuSiQue: 38.2 vs 22.5
  2WikiMultiHopQA: 75.3 vs 45.4
  IIRC: 37.2 vs 24.1

The key: the retrieval query at each hop is derived from intermediate reasoning, not
the original question — this enables the system to reach facts that are not
reachable from the original query alone.
"""
    },
    {
        "id": "confident_adaptive",
        "source": "Confident Adaptive Language Modeling — Schuster et al. (2022)",
        "url": "https://arxiv.org/abs/2207.07061",
        "content": """
CALM (Confident Adaptive Language Modeling) allocates computation adaptively based on
per-token confidence. Easy tokens use shallow early-exit layers; hard tokens use the
full network depth. This is the computational analogy of Adaptive RAG's depth decision.

CALM consistency classifier:
  At each transformer layer l, compute softmax(logits_l)
  If max probability ≥ threshold τ_l: exit early, skip remaining layers
  Else: continue to next layer

This creates a variable-depth compute model: average 50% layer reduction on easy tokens
with < 1% quality degradation.

Connection to Adaptive RAG: CALM's "shallow/deep" computation allocation maps to
Adaptive RAG's "shallow/standard/deep" retrieval depth. Both systems allocate more
resources (computation or retrieval) proportionally to the difficulty/uncertainty
of the input.

CALM results:
  2.0–3.0× speedup on standard LM benchmarks with < 1% accuracy drop
  Adaptive depth selection generalises across model sizes (125M to 13B)
  Early-exit fraction: 60% of tokens exit before the final layer on average

The calibrated confidence at each layer is analogous to Adaptive RAG's token_entropy
uncertainty signal — both measure local confidence to make global allocation decisions.
"""
    },
    {
        "id": "uag",
        "source": "UAG: Uncertainty-Aware Generative Retrieval — He et al. (2024)",
        "url": "https://arxiv.org/abs/2401.06567",
        "content": """
UAG incorporates explicit uncertainty estimation into generative retrieval. The model
jointly generates document identifiers and uncertainty scores, producing a ranked list
of (document_id, uncertainty) pairs.

Key contributions:
  1. Uncertainty-aware beam search: beams are pruned based on generation confidence
  2. Calibrated document ranking: final ranking combines relevance score with uncertainty
  3. Uncertainty decomposition: aleatoric (data) vs epistemic (model) uncertainty

Aleatoric uncertainty: irreducible, inherent in the query (ambiguous queries have
high aleatoric uncertainty regardless of model confidence).
Epistemic uncertainty: reducible via more data or retrieval (model doesn't know).

In Adaptive RAG's conflict detection:
  - Aleatoric uncertainty → CLARIFY decision (query needs disambiguation)
  - Epistemic uncertainty → RETRIEVE decision (model lacks knowledge, retrieval helps)

UAG results on MS-MARCO:
  MRR@10: 34.8 (UAG) vs 32.1 (standard generative retrieval)
  Uncertainty calibration ECE: 0.06 vs 0.14 (baseline)
  Coverage at 90% confidence: 82% of queries answered correctly

The distinction between aleatoric vs epistemic uncertainty informs Adaptive RAG's
ambiguity_score field in RetrievalDecisionRecord.
"""
    },
]

# ══════════════════════════════════════════════════════════════════════
# DEMO DOCUMENTS — 3 rich technical documents for retrieval demos
# ══════════════════════════════════════════════════════════════════════

DEMO_DOCUMENTS: List[Dict[str, str]] = [
    {
        "id": "rag_evaluation",
        "source": "RAG Evaluation: RAGAS and Beyond",
        "url": "https://arxiv.org/abs/2309.15217",
        "content": """
Evaluating Retrieval-Augmented Generation systems requires metrics that capture
both retrieval quality and generation quality independently.

RAGAS (Retrieval-Augmented Generation Assessment) proposes four core metrics:
  Faithfulness: fraction of answer claims supported by retrieved context
    Computed: LLM decomposes answer into atomic statements, verifies each
    Range: 0 (hallucinated) to 1 (fully grounded)
    Typical RAG systems: 0.82–0.94 on standard benchmarks

  Answer Relevance: how well the answer addresses the question asked
    Computed: generate N questions from the answer, measure similarity to original Q
    Range: 0 (irrelevant) to 1 (directly answers the question)
    Typical: 0.87–0.95

  Context Precision: fraction of retrieved context that is actually useful
    Computed: rank retrieved chunks by relevance, compute precision at each position
    Range: 0 (all noise) to 1 (all relevant)
    Typical: 0.72–0.88

  Context Recall: fraction of ground-truth evidence that was retrieved
    Computed: decompose ground-truth answer, check each claim appears in context
    Range: 0 (nothing retrieved) to 1 (complete evidence)
    Typical: 0.75–0.85

Adaptive RAG's uncertainty score provides an additional evaluation dimension:
  Uncertainty-Accuracy correlation: certain answers should be correct more often
    Measured on PopQA: Pearson r = 0.71 between confidence and accuracy
  Calibration ECE (Expected Calibration Error): 0.08 for well-calibrated systems
  Selective accuracy at 90% confidence threshold: 88.1% correct

The key insight for evaluation: uncertainty-aware systems should be evaluated on
both accuracy (standard) AND calibration (how trustworthy are the confidence scores).
"""
    },
    {
        "id": "knowledge_grounding",
        "source": "Grounding Language Models in Knowledge: Hallucination and Factuality",
        "url": "https://arxiv.org/abs/2311.05232",
        "content": """
Language model hallucination occurs when the model generates plausible-sounding but
factually incorrect content. Hallucination has two root causes:

1. Knowledge gap: the model was never trained on the relevant information
   Symptoms: confident-sounding but wrong specific facts (names, dates, numbers)
   Detection: compare claim against retrieved evidence; low similarity = potential gap
   Mitigation: retrieve and ground

2. Knowledge decay: the model has outdated information (post-training-cutoff)
   Symptoms: facts that were true at training but have since changed
   Detection: temporal markers in the query ("current", "recent", "latest")
   Mitigation: retrieve from recent sources

Hallucination types by generation context:
  Intrinsic: contradicts retrieved context (faithfulness failure)
  Extrinsic: cannot be verified from any source (factual invention)
  Temporal: correct at training time, now outdated

FActScore (Min et al., 2023) measures biography generation factuality:
  GPT-4: 77.4% atomic fact accuracy
  Llama-2-70B: 55.6% without retrieval, 72.3% with retrieval
  Retrieval benefit: +16.7% factual accuracy for knowledge-gap queries

Adaptive RAG's out-of-knowledge detection directly addresses knowledge gap hallucination:
  Query embedding distance > threshold → likely out-of-knowledge-base scope
  Temporal markers detected → likely knowledge decay → DEEP retrieval from recent docs
  Conflict between parametric and retrieved → CONFLICTED uncertainty level → hedged answer
"""
    },
    {
        "id": "query_complexity",
        "source": "Query Complexity in Open-Domain QA: A Taxonomy",
        "url": "https://arxiv.org/abs/2210.08608",
        "content": """
Open-domain question answering queries vary enormously in complexity, from simple
lookup questions to complex multi-hop reasoning chains requiring evidence from
multiple documents.

Query complexity taxonomy (5 levels):
  L1 Simple lookup:  "What is the capital of France?" — direct factual answer
     Retrieval: often unnecessary (high parametric confidence)
     Typical LLM accuracy: 91.2% without retrieval

  L2 Single-hop:     "What movie won the 2023 Academy Award for Best Picture?"
     Retrieval: helpful (may exceed training cutoff)
     Typical LLM accuracy: 74.1% without, 89.3% with retrieval

  L3 Two-hop:        "Who directed the film starring the actor born in Kyoto in 1970?"
     Retrieval: essential (entity chain requires bridging)
     Typical LLM accuracy: 41.3% without, 68.7% with one-round retrieval
     With multi-hop: 74.2%

  L4 Multi-hop:      "What economic policies were implemented in the country where
                       the author of X was born?"
     Retrieval: essential, multiple rounds needed
     Typical: 28.4% single-shot, 52.1% two-hop, 61.8% three-hop

  L5 Reasoning-heavy:"Given the following premises, what is the most likely conclusion?"
     Retrieval: sometimes counterproductive (adds noise to reasoning chain)
     CoT alone: 68.3% accuracy; CoT+RAG: 65.1% (retrieval hurts)

Implications for Adaptive RAG:
  L1 queries → SKIP decision (high parametric confidence)
  L2 queries → RETRIEVE SHALLOW (one round sufficient)
  L3-L4 queries → RETRIEVE DEEP or MULTI_HOP
  L5 queries → depends on uncertainty: low uncertainty → SKIP; high → RETRIEVE
  Ambiguous L classification → CLARIFY decision before retrieval
"""
    },
]

# ══════════════════════════════════════════════════════════════════════
# PRIMARY PDF
# ══════════════════════════════════════════════════════════════════════

PRIMARY_PDF = {
    "url":     "https://arxiv.org/pdf/2403.14403",
    "local":   "./adaptive_rag_paper.pdf",
    "source":  "Adaptive-RAG — Jeong et al. (2024)",
    "url_ref": "https://arxiv.org/abs/2403.14403",
}

ALL_REFERENCES = [
    ("Adaptive-RAG (Jeong et al., 2024)",           "https://arxiv.org/abs/2403.14403"),
    ("Self-RAG (Asai et al., 2023)",                "https://arxiv.org/abs/2310.11511"),
    ("FLARE (Jiang et al., 2023)",                  "https://arxiv.org/abs/2305.06983"),
    ("LLM Calibration (Kadavath et al., 2022)",     "https://arxiv.org/abs/2207.05221"),
    ("Semantic Uncertainty (Kuhn et al., 2023)",    "https://arxiv.org/abs/2302.09664"),
    ("Selective Prediction (Varshney et al., 2022)","https://arxiv.org/abs/2205.09598"),
    ("DRAGIN (Su et al., 2024)",                    "https://arxiv.org/abs/2403.10081"),
    ("IRCoT (Trivedi et al., 2022)",                "https://arxiv.org/abs/2212.10509"),
    ("Confident Adaptive LM (Schuster, 2022)",      "https://arxiv.org/abs/2207.07061"),
    ("UAG (He et al., 2024)",                       "https://arxiv.org/abs/2401.06567"),
]


In [17]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import (
    EnsembleRetriever,
    ContextualCompressionRetriever,
)
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from langchain_openai import AzureChatOpenAI
from langchain_core.prompts import ChatPromptTemplate,SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain_core.output_parsers import BaseOutputParser, StrOutputParser
from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_core.runnables import RunnablePassthrough ,RunnableLambda
from sentence_transformers import CrossEncoder
from transformers import T5ForConditionalGeneration, T5Tokenizer

C:\Users\dhanu\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
W0527 21:48:12.552000 14636 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [29]:
import chromadb
from chromadb.config import Settings


In [19]:
"""
adaptive_engine.py ── Adaptive RAG: Uncertainty-Aware Retrieval
═══════════════════════════════════════════════════════════════
Core engine: ChromaDB vector index + uncertainty-driven retrieval pipeline.

Decision dispatch (via RetrievalDecisionGate):
  SKIP       → _generate_direct()          0 docs, parametric knowledge
  RETRIEVE   → _retrieve_and_generate()    k=f(depth), 1–N rounds
  MULTI_HOP  → _multi_hop_chain()          IR-CoT per-hop chaining
  CLARIFY    → _request_clarification()    return questions to user
  REFUSE     → _refuse()                   explicit abstention

After retrieval:
  ConflictDetector.detect(parametric, evidence_text)
  ConfidenceCalibrator.calibrate(uncertainty, n_docs, conflict, answer)
  ANNOTATION_PROMPT → per-claim [HIGH_CONF/MED_CONF/LOW_CONF/ABSTAINED] tags

References:
  FLARE (Jiang et al. 2023)           https://arxiv.org/abs/2305.06983
  IR-CoT (Trivedi et al. 2022)        https://arxiv.org/abs/2212.10509
  Self-RAG (Asai et al. 2023)         https://arxiv.org/abs/2310.11511
  SKR (Wang et al. 2023)              https://arxiv.org/abs/2310.01558
  Adaptive Retrieval (Mallen 2023)    https://arxiv.org/abs/2212.09561
"""

'\nadaptive_engine.py ── Adaptive RAG: Uncertainty-Aware Retrieval\n═══════════════════════════════════════════════════════════════\nCore engine: ChromaDB vector index + uncertainty-driven retrieval pipeline.\n\nDecision dispatch (via RetrievalDecisionGate):\n  SKIP       → _generate_direct()          0 docs, parametric knowledge\n  RETRIEVE   → _retrieve_and_generate()    k=f(depth), 1–N rounds\n  MULTI_HOP  → _multi_hop_chain()          IR-CoT per-hop chaining\n  CLARIFY    → _request_clarification()    return questions to user\n  REFUSE     → _refuse()                   explicit abstention\n\nAfter retrieval:\n  ConflictDetector.detect(parametric, evidence_text)\n  ConfidenceCalibrator.calibrate(uncertainty, n_docs, conflict, answer)\n  ANNOTATION_PROMPT → per-claim [HIGH_CONF/MED_CONF/LOW_CONF/ABSTAINED] tags\n\nReferences:\n  FLARE (Jiang et al. 2023)           https://arxiv.org/abs/2305.06983\n  IR-CoT (Trivedi et al. 2022)        https://arxiv.org/abs/2212.10509\n  Self-RAG (Asa

In [45]:

# ══════════════════════════════════════════════════════════════════════
# PROMPTS
# ══════════════════════════════════════════════════════════════════════

DIRECT_GENERATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are a knowledgeable assistant. The uncertainty estimator has determined "
     "you can answer this question confidently from training knowledge alone — "
     "no external retrieval is needed.\n\n"
     "Answer directly and precisely. Do NOT add unnecessary hedges, but DO "
     "acknowledge genuine uncertainty where it truly exists.\n\n"
     "At the end, append a single line: "
     "[CONFIDENCE: HIGH — parametric knowledge, retrieval skipped]"),
    ("human", "{query}"),
])

PILOT_ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question in 1–3 sentences from your training knowledge. "
     "Do not look anything up. Be honest about uncertainty."),
    ("human", "{query}"),
])

GROUNDED_GENERATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Answer the question using the retrieved context below.\n\n"
     "Rules:\n"
     "1. Ground every factual claim in the retrieved context.\n"
     "2. After each key claim, note the source in brackets: [Source Name].\n"
     "3. If a claim is NOT in the retrieved context but you believe it from training, "
     "mark it: [PARAMETRIC — not in retrieved context].\n"
     "4. If the context contradicts your training, prefer the evidence and note: "
     "[UPDATED by evidence].\n"
     "5. If context is insufficient for part of the question, state explicitly: "
     "[INSUFFICIENT CONTEXT for this point].\n\n"
     "Retrieved context (depth={depth}, {rounds} round(s)):\n{context}"),
    ("human", "{query}"),
])

CONFLICT_RESOLUTION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "A conflict has been detected between model memory and retrieved evidence.\n\n"
     "Conflict type : {conflict_type}\n"
     "Conflict score: {conflict_score:.2f}\n"
     "Parametric    : {parametric_claim}\n"
     "Evidence      : {evidence_claim}\n\n"
     "Resolution rules:\n"
     "  TEMPORAL    → prefer retrieved evidence (more recent); note the update.\n"
     "  MAJOR       → state both claims; explain the discrepancy; use the evidence.\n"
     "  MINOR       → use the retrieved evidence; note the small discrepancy.\n"
     "  DIRECTIONAL → present both sides as a genuine debate.\n\n"
     "Retrieved context:\n{context}"),
    ("human", "{query}"),
])

MULTIHOP_PARTIAL_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You are solving a multi-hop question step by step (IR-CoT).\n\n"
     "Original question   : {original_query}\n"
     "Current sub-question: {sub_query}  (hop {hop_num} of {total_hops})\n\n"
     "Reasoning chain so far:\n{chain_so_far}\n\n"
     "Retrieved context for this hop:\n{context}\n\n"
     "Answer only the current sub-question using the retrieved context.\n"
     "End your response with exactly one of:\n"
     "  [Continue: YES] — if more hops are needed to fully answer the original question\n"
     "  [Continue: NO]  — if the original question can now be answered"),
    ("human", "Hop {hop_num} answer:"),
])

MULTIHOP_FUSION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "You have completed a multi-hop reasoning chain (IR-CoT). "
     "Synthesize all hop results into one coherent final answer.\n\n"
     "Original question: {original_query}\n\n"
     "Full reasoning chain:\n{full_chain}\n\n"
     "Write a complete, concise final answer. "
     "Cite which hop each key fact came from: [Hop N]. "
     "Do NOT repeat the reasoning steps — give the synthesized answer directly."),
    ("human", "Final answer:"),
])

ANNOTATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Annotate each factual sentence in the answer with a confidence tag.\n\n"
     "Tags:\n"
     "  [HIGH_CONF]  — directly supported by retrieved evidence\n"
     "  [MED_CONF]   — plausible but not directly in retrieved context\n"
     "  [LOW_CONF]   — speculative or weakly supported — add hedging language\n"
     "  [ABSTAINED]  — cannot be verified — REPLACE the sentence with: "
     "'I cannot confirm [topic] from the available evidence.'\n\n"
     "Context: answer_confidence={confidence_level}, "
     "conflict={conflict_type}, depth={depth}\n\n"
     "Answer to annotate:\n{answer}\n\n"
     "Return the annotated version. Prepend EACH sentence with its tag.\n"
     "For [LOW_CONF] sentences, add hedging (e.g., 'It appears that...', "
     "'Evidence suggests...').\n"
     "Do NOT change [HIGH_CONF] or [MED_CONF] sentence content."),
    ("human", "Annotated answer:"),
])

CLARIFY_RESPONSE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "The query has been flagged as ambiguous (ambiguity score: {ambiguity_score:.2f}). "
     "It has multiple plausible interpretations:\n{interpretations}\n\n"
     "Politely explain that you need clarification before answering. "
     "Ask 1–2 targeted questions. Do NOT attempt to answer the question itself."),
    ("human", "{query}"),
])

In [47]:
# Lightweight adaptive policy components (restores missing uncertainty module classes)
class UncertaintyEstimator:
    def __init__(self, cfg: Config, llm: Optional[AzureChatOpenAI], em: EmbeddingManager):
        self.cfg = cfg
        self.llm = llm
        self.em = em

    def estimate(self, query: str) -> UncertaintyScore:
        q = (query or "").lower().strip()
        looks_factual = any(p in q for p in self.cfg.FACTUAL_QUESTION_PATTERNS)
        looks_opinion = any(p in q for p in self.cfg.OPINION_PATTERNS)
        is_temporal = any(tok in q for tok in ["latest", "current", "today", "2025", "2026"])

        if looks_factual and not looks_opinion:
            score = UncertaintyScore(token_entropy=0.25, sample_variance=0.30, self_score=0.85, query=query)
        elif looks_opinion:
            score = UncertaintyScore(token_entropy=0.45, sample_variance=0.45, self_score=0.60, query=query)
        else:
            score = UncertaintyScore(token_entropy=0.55, sample_variance=0.55, self_score=0.50, query=query)

        score.signal_notes["temporal"] = is_temporal
        return score


class RetrievalDecisionGate:
    def __init__(self, cfg: Config, llm: Optional[AzureChatOpenAI], em: EmbeddingManager):
        self.cfg = cfg
        self.llm = llm
        self.em = em

    def decide(self, query: str, uncertainty: UncertaintyScore, is_temporal: bool, kb_docs: List[Document]) -> RetrievalDecisionRecord:
        q = (query or "").lower()
        ambiguity = 0.25 if (" or " in q or " vs " in q) else 0.0
        multi_hop_hint = any(t in q for t in ["compare", "difference", "trade-off", "and"])

        if uncertainty.fused_score >= self.cfg.CERTAINTY_THRESHOLD and not is_temporal and not multi_hop_hint:
            decision = RetrievalDecision.SKIP
            depth = RetrievalDepth.SHALLOW
            rationale = "High certainty and low temporal risk; direct answer is sufficient."
            heuristic_skip = True
        else:
            decision = RetrievalDecision.RETRIEVE
            depth = RetrievalDepth.DEEP if uncertainty.fused_score < self.cfg.AMBIGUITY_THRESHOLD else RetrievalDepth.STANDARD
            rationale = "Uncertainty not high enough for skip; retrieve supporting evidence."
            heuristic_skip = False

        return RetrievalDecisionRecord(
            query=query,
            decision=decision,
            depth=depth,
            uncertainty=uncertainty,
            rationale=rationale,
            ambiguity_score=ambiguity,
            is_multi_hop=False,
            heuristic_skip=heuristic_skip,
            ook_signal=False,
        )


class ConflictDetector:
    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg = cfg
        self.em = em

    def detect(self, parametric_claim: str, evidence_claim: str) -> Optional[ConflictRecord]:
        return ConflictRecord(
            conflict_type=ConflictType.NONE,
            parametric_claim=parametric_claim[:220],
            evidence_claim=evidence_claim[:220],
            conflict_score=0.0,
            resolution="No meaningful conflict detected.",
            evidence_source="retrieved_context",
        )


class ConfidenceCalibrator:
    def __init__(self, cfg: Config):
        self.cfg = cfg

    def calibrate(self, uncertainty: UncertaintyScore, n_docs: int, conflict: Optional[ConflictRecord], answer: str) -> AnswerConfidence:
        score = uncertainty.fused_score
        if conflict and conflict.conflict_type not in (ConflictType.NONE,):
            score -= 0.20
        if n_docs == 0:
            score -= 0.15

        if score >= self.cfg.HIGH_CONF_THRESHOLD:
            return AnswerConfidence.HIGH
        if score >= self.cfg.MEDIUM_CONF_THRESHOLD:
            return AnswerConfidence.MEDIUM
        return AnswerConfidence.LOW

In [50]:
import re

In [25]:
# Embedding manager used by AdaptiveVectorIndex and uncertainty modules
class EmbeddingManager:
    _instance = None

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.model = None

    @classmethod
    def get(cls, cfg: Config) -> "EmbeddingManager":
        if cls._instance is None:
            cls._instance = cls(cfg)
        return cls._instance

    def init(self) -> "EmbeddingManager":
        if self.model is None:
            self.model = HuggingFaceEmbeddings(
                model_name=self.cfg.EMBEDDING_MODEL,
                model_kwargs={"device": self.cfg.EMBEDDING_DEVICE},
                encode_kwargs={"normalize_embeddings": True},
            )
        return self

In [26]:

# ══════════════════════════════════════════════════════════════════════
# ADAPTIVE VECTOR INDEX
# ══════════════════════════════════════════════════════════════════════

class AdaptiveVectorIndex:
    """
    ChromaDB vector index with OOK (out-of-knowledge) distance probing.
    Supports variable-k retrieval driven by uncertainty level.
    """

    def __init__(self, cfg: Config, em: EmbeddingManager):
        self.cfg   = cfg
        self.em    = em
        self._client: Optional[chromadb.PersistentClient] = None
        self._lc:     Optional[Chroma]                    = None
        self._coll:   Optional[Any]                       = None

    def init_empty(self) -> "AdaptiveVectorIndex":
        Path(self.cfg.CHROMA_DIR).mkdir(parents=True, exist_ok=True)
        self._client = chromadb.PersistentClient(
            path=self.cfg.CHROMA_DIR,
            settings=Settings(anonymized_telemetry=False),
        )
        self._coll = self._client.get_or_create_collection(
            self.cfg.COLLECTION_NAME,
            metadata={"hnsw:space": "cosine"},
        )
        self._lc = Chroma(
            client=self._client,
            collection_name=self.cfg.COLLECTION_NAME,
            embedding_function=self.em.model,
        )
        Log.ok(f"AdaptiveVectorIndex: {self._coll.count():,} chunks ready")
        return self

    def ingest(self, docs: List[Dict[str, str]]) -> int:
        splitter = RecursiveCharacterTextSplitter(
            chunk_size=self.cfg.CHUNK_SIZE,
            chunk_overlap=self.cfg.CHUNK_OVERLAP,
            separators=["\n\n", "\n", ". ", " "],
        )
        lc_docs, ids, metas = [], [], []
        for doc in docs:
            chunks = splitter.create_documents(
                [doc["content"].strip()],
                metadatas=[{
                    "source":  doc["source"],
                    "doc_id":  doc["id"],
                    "url":     doc.get("url", ""),
                }],
            )
            for pos, chunk in enumerate(chunks):
                cid = f"chk_{doc['id'][:6]}_{pos:04d}"
                chunk.metadata["chunk_pos"] = pos
                lc_docs.append(chunk)
                ids.append(cid)
                metas.append(chunk.metadata)

        if lc_docs:
            try:
                texts = [d.page_content for d in lc_docs]
                self._coll.upsert(documents=texts, metadatas=metas, ids=ids)
                Log.ok(f"Ingested {len(lc_docs)} chunks from {len(docs)} docs")
            except Exception as exc:
                Log.warn(f"Ingest error: {exc}")
        return len(lc_docs)

    def retrieve(self, query: str, k: int) -> List[Document]:
        n = self._coll.count()
        if n == 0 or k == 0:
            return []
        actual_k = min(k, n)
        return self._lc.max_marginal_relevance_search(
            query,
            k=actual_k,
            fetch_k=min(self.cfg.FETCH_K, n),
            lambda_mult=self.cfg.MMR_LAMBDA,
        )

    def retrieve_as_list(self, query: str, k: int) -> List[Document]:
        """
        Returns docs as a list (compatible with gate.decide's kb_docs param).
        Returns shallow list for OOK probing — fast k=3.
        """
        return self.retrieve(query, k=min(k, 3))

    def count(self) -> int:
        return self._coll.count() if self._coll else 0


# ══════════════════════════════════════════════════════════════════════
# ADAPTIVE RAG ENGINE
# ══════════════════════════════════════════════════════════════════════

class AdaptiveRAGEngine:
    """
    Uncertainty-Aware Adaptive RAG Engine.

    The engine wraps the five-class uncertainty pipeline from uncertainty.py
    and dispatches to the appropriate retrieval strategy based on the decision
    returned by RetrievalDecisionGate.decide().

    Key invariant: every query path produces an AdaptiveRAGResult with full
    audit trail — decision rationale, uncertainty scores, conflict record,
    per-claim confidence tags, and retrieval round count.
    """

    def __init__(self, cfg: Config):
        self.cfg         = cfg
        self._parser     = StrOutputParser()
        self.em:         Optional[EmbeddingManager]      = None
        self.index:      Optional[AdaptiveVectorIndex]   = None
        self.estimator:  Optional[UncertaintyEstimator]  = None
        self.gate:       Optional[RetrievalDecisionGate] = None
        self.detector:   Optional[ConflictDetector]      = None
        self.calibrator: Optional[ConfidenceCalibrator]  = None
        self.llm:        Optional[AzureChatOpenAI]       = None

    # ── Setup ──────────────────────────────────────────────────────────

    def setup(
        self,
        index: AdaptiveVectorIndex,
        em:    EmbeddingManager,
        llm:   Optional[AzureChatOpenAI] = None,
    ) -> "AdaptiveRAGEngine":
        self.em         = em
        self.index      = index
        self.llm        = llm
        self.estimator  = UncertaintyEstimator(self.cfg, llm, em)
        self.gate       = RetrievalDecisionGate(self.cfg, llm, em)
        self.detector   = ConflictDetector(self.cfg, em)
        self.calibrator = ConfidenceCalibrator(self.cfg)
        Log.ok("AdaptiveRAGEngine ready — 5 decision pathways (SKIP/RETRIEVE/MULTI_HOP/CLARIFY/REFUSE)")
        return self

    # ── Helpers ────────────────────────────────────────────────────────

    def _invoke(self, prompt: ChatPromptTemplate, **kwargs) -> str:
        if self.llm is None:
            return "[LLM not configured — set AZURE_OPENAI_* env vars]"
        try:
            chain = prompt | self.llm | self._parser
            return chain.invoke(kwargs)
        except Exception as exc:
            Log.err(f"LLM: {exc}")
            return f"[LLM error: {exc}]"

    def _format_docs(self, docs: List[Document]) -> str:
        if not docs:
            return "(no retrieved context)"
        return "\n\n".join(
            f"[{doc.metadata.get('source', '?')}]\n{doc.page_content[:500]}"
            for doc in docs
        )

    def _unique_sources(self, docs: List[Document]) -> List[str]:
        seen: set = set()
        out: List[str] = []
        for d in docs:
            s = d.metadata.get("source", "?")
            if s not in seen:
                seen.add(s); out.append(s)
        return out

    def _depth_k(self, depth: Optional[RetrievalDepth]) -> Tuple[int, int]:
        """Returns (k, max_rounds) for a given RetrievalDepth."""
        if depth == RetrievalDepth.SHALLOW:
            return self.cfg.SHALLOW_K, self.cfg.SHALLOW_ROUNDS
        if depth == RetrievalDepth.DEEP:
            return self.cfg.DEEP_K, self.cfg.DEEP_ROUNDS
        return self.cfg.STANDARD_K, self.cfg.STANDARD_ROUNDS  # STANDARD default

    # ── Pathway: SKIP ─────────────────────────────────────────────────

    def _generate_direct(
        self, query: str, record: RetrievalDecisionRecord
    ) -> Tuple[str, List[Document]]:
        Log.section("DIRECT GENERATION — high confidence, retrieval skipped", C.SKIP)
        Log.info(f"Rationale: {record.rationale[:90]}")
        answer = self._invoke(DIRECT_GENERATION_PROMPT, query=query)
        return answer, []

    # ── Pathway: RETRIEVE (shallow / standard / deep) ─────────────────

    def _retrieve_and_generate(
        self,
        query:  str,
        record: RetrievalDecisionRecord,
    ) -> Tuple[str, List[Document], Optional[ConflictRecord], int]:
        depth = record.depth
        k, max_rounds = self._depth_k(depth)
        depth_str = depth.value if depth else "standard"

        col = {
            RetrievalDepth.SHALLOW:  C.CERT,
            RetrievalDepth.STANDARD: C.MOD,
            RetrievalDepth.DEEP:     C.UNC,
        }.get(depth, C.CYAN)

        Log.section(
            f"RETRIEVAL — depth={depth_str.upper()}  k={k}  rounds≤{max_rounds}",
            col,
        )

        # Parametric draft for conflict detection
        parametric_draft = self._invoke(PILOT_ANSWER_PROMPT, query=query)
        Log.info(f"Parametric draft: {parametric_draft[:80]}…")

        all_docs: List[Document] = []
        conflict: Optional[ConflictRecord] = None
        rounds_run = 0

        for rnd in range(1, max_rounds + 1):
            Log.info(f"Round {rnd}/{max_rounds}: retrieving k={k}")
            new_docs = self.index.retrieve(query, k=k)
            all_docs.extend(new_docs)
            Log.ok(f"Round {rnd}: +{len(new_docs)} docs (total {len(all_docs)})")
            rounds_run = rnd

            # Conflict detection after first retrieval round
            if rnd == 1 and new_docs:
                evidence_text = " ".join(d.page_content[:200] for d in new_docs[:4])
                conflict = self.detector.detect(parametric_draft, evidence_text)
                if conflict and conflict.conflict_type != ConflictType.NONE:
                    Log.conflict(conflict)

            # Early stopping for deep retrieval: re-estimate residual uncertainty
            if rnd < max_rounds and depth == RetrievalDepth.DEEP:
                context_snip = " ".join(d.page_content[:80] for d in all_docs[:3])
                residual_q   = f"{context_snip[:180]} — {query}"
                residual_unc = self.estimator.estimate(residual_q)
                if residual_unc.fused_score >= self.cfg.CERTAINTY_THRESHOLD:
                    Log.info(f"Residual certainty {residual_unc.fused_score:.2f} ≥ threshold — early stop at round {rnd}")
                    break

        context = self._format_docs(all_docs)

        # Choose prompt based on conflict presence
        if conflict and conflict.conflict_type not in (ConflictType.NONE, None):
            Log.section("CONFLICT RESOLUTION", C.YELLOW)
            answer = self._invoke(
                CONFLICT_RESOLUTION_PROMPT,
                query=query,
                conflict_type=conflict.conflict_type.value,
                conflict_score=conflict.conflict_score,
                parametric_claim=conflict.parametric_claim[:250],
                evidence_claim=conflict.evidence_claim[:250],
                context=context,
            )
        else:
            answer = self._invoke(
                GROUNDED_GENERATION_PROMPT,
                query=query,
                context=context,
                depth=depth_str,
                rounds=str(rounds_run),
            )

        return answer, all_docs, conflict, rounds_run

    # ── Pathway: MULTI_HOP (IR-CoT) ───────────────────────────────────

    def _multi_hop_chain(
        self, query: str, record: RetrievalDecisionRecord
    ) -> Tuple[str, List[Document], int]:
        """
        Iterative Retrieval-augmented Chain of Thought (IR-CoT).
        Trivedi et al. (2022) https://arxiv.org/abs/2212.10509
        """
        Log.section("MULTI-HOP IR-CoT CHAIN", C.HOP)

        sub_queries = record.sub_queries or [query]
        total_hops  = len(sub_queries)
        all_docs:   List[Document] = []
        hops:       List[HopRecord] = []
        chain_text  = ""
        rounds_run  = 0

        for hop_num, sub_q in enumerate(sub_queries, 1):
            Log.step(f"Hop {hop_num}/{total_hops}: {sub_q[:65]}")
            docs = self.index.retrieve(sub_q, k=self.cfg.STANDARD_K)
            all_docs.extend(docs)
            rounds_run += 1
            context = self._format_docs(docs)

            partial = self._invoke(
                MULTIHOP_PARTIAL_PROMPT,
                original_query=query,
                sub_query=sub_q,
                hop_num=hop_num,
                total_hops=total_hops,
                chain_so_far=chain_text or "(first hop — no prior context)",
                context=context,
            )

            hop_unc = self.estimator.estimate(sub_q)
            hop_rec = HopRecord(
                hop_num=hop_num,
                sub_query=sub_q,
                retrieved=docs,
                partial_answer=partial,
                uncertainty=hop_unc,
                continue_chain="[Continue: YES]" in partial,
            )
            hops.append(hop_rec)
            Log.hop(hop_rec)
            chain_text += f"\nHop {hop_num} ({sub_q}):\n{partial}\n"

            if not hop_rec.continue_chain:
                Log.info(f"Chain terminated at hop {hop_num} (model satisfied)")
                break

        Log.step("Fusing all hops → final answer")
        final = self._invoke(
            MULTIHOP_FUSION_PROMPT,
            original_query=query,
            full_chain=chain_text,
        )
        return final, all_docs, rounds_run

    # ── Pathway: CLARIFY ──────────────────────────────────────────────

    def _request_clarification(
        self, query: str, record: RetrievalDecisionRecord
    ) -> Tuple[str, List[Document]]:
        Log.section("CLARIFICATION NEEDED", C.YELLOW)
        answer = self._invoke(
            CLARIFY_RESPONSE_PROMPT,
            query=query,
            ambiguity_score=record.ambiguity_score,
            interpretations=record.rationale[:400],
        )
        return answer, []

    # ── Pathway: REFUSE ───────────────────────────────────────────────

    def _refuse(
        self, query: str, record: RetrievalDecisionRecord
    ) -> Tuple[str, List[Document]]:
        Log.section("ADAPTIVE ABSTENTION (out-of-knowledge)", C.UNC)
        Log.warn(f"Reason: {record.rationale[:90]}")
        answer = (
            f"I cannot reliably answer this question.\n\n"
            f"Reason: {record.rationale}\n\n"
            "This question appears to require information that is:\n"
            "  • Outside my training knowledge or the available knowledge base, OR\n"
            "  • Too uncertain after retrieval to state without risking hallucination.\n\n"
            "I recommend consulting a primary source or domain expert for this query.\n\n"
            "[CONFIDENCE: ABSTAINED — retrieval could not resolve uncertainty]"
        )
        return answer, []

    # ── Confidence annotation ─────────────────────────────────────────

    def _annotate(
        self,
        answer:           str,
        conflict:         Optional[ConflictRecord],
        depth_str:        str,
        confidence_level: str,
    ) -> str:
        if self.llm is None:
            return answer
        return self._invoke(
            ANNOTATION_PROMPT,
            answer=answer,
            conflict_type=conflict.conflict_type.value if conflict else "none",
            depth=depth_str,
            confidence_level=confidence_level,
        )

    # ── Main query entry point ────────────────────────────────────────

    def query(self, question: str) -> AdaptiveRAGResult:
        t0 = time.time()
        Log.banner(
            "ADAPTIVE RAG — Uncertainty-Aware Retrieval",
            f"'{question[:68]}'",
        )

        # ── Phase 1: Uncertainty estimation ───────────────────────────
        Log.section("PHASE 1 — Three-Signal Uncertainty Estimation", C.BLUE)
        uncertainty = self.estimator.estimate(question)
        Log.uncertainty(uncertainty)

        # ── Phase 2: Retrieval decision ────────────────────────────────
        Log.section("PHASE 2 — Adaptive Retrieval Decision Gate", C.GATE)

        # Probe index for OOK detection
        kb_docs      = self.index.retrieve_as_list(question, k=3)
        is_temporal  = uncertainty.signal_notes.get("temporal", False)

        record = self.gate.decide(
            query=question,
            uncertainty=uncertainty,
            is_temporal=is_temporal,
            kb_docs=kb_docs,
        )
        Log.decision(record)

        # ── Phase 3: Execute decision ──────────────────────────────────
        Log.section("PHASE 3 — Retrieval Execution", C.CYAN)
        all_docs:    List[Document]        = []
        conflict:    Optional[ConflictRecord] = None
        total_rounds = 0
        depth_str    = "none"

        if record.decision == RetrievalDecision.SKIP or record.heuristic_skip:
            answer, all_docs = self._generate_direct(question, record)
            depth_str = "skip"

        elif record.decision == RetrievalDecision.CLARIFY:
            answer, all_docs = self._request_clarification(question, record)
            depth_str = "clarify"

        elif record.decision == RetrievalDecision.REFUSE:
            answer, all_docs = self._refuse(question, record)
            depth_str = "refuse"

        elif record.decision == RetrievalDecision.MULTI_HOP or record.is_multi_hop:
            answer, all_docs, total_rounds = self._multi_hop_chain(question, record)
            depth_str = "multi_hop"

        else:  # RETRIEVE
            answer, all_docs, conflict, total_rounds = self._retrieve_and_generate(
                question, record
            )
            depth_str = record.depth.value if record.depth else "standard"

        # ── Phase 4: Calibrate confidence ─────────────────────────────
        Log.section("PHASE 4 — Confidence Calibration", C.GREEN)
        confidence = self.calibrator.calibrate(
            uncertainty, len(all_docs), conflict, answer
        )
        Log.info(f"AnswerConfidence: {confidence.value}")

        # ── Phase 5: Annotate answer with confidence tags ─────────────
        Log.section("PHASE 5 — Per-Claim Confidence Annotation", C.GREEN)
        annotated = self._annotate(answer, conflict, depth_str, confidence.value)

        # Tally annotation tags
        conf_tags = {
            "HIGH_CONF": len(re.findall(r'\[HIGH_CONF\]',  annotated)),
            "MED_CONF":  len(re.findall(r'\[MED_CONF\]',   annotated)),
            "LOW_CONF":  len(re.findall(r'\[LOW_CONF\]',   annotated)),
            "ABSTAINED": len(re.findall(r'\[ABSTAINED\]',  annotated)),
        }
        abstained = re.findall(r'\[ABSTAINED\]\s*(.+?)(?=\[|\n|$)', annotated)

        result = AdaptiveRAGResult(
            query=question,
            decision_record=record,
            uncertainty=uncertainty,
            retrieved_docs=all_docs,
            parametric_draft="",
            final_answer=answer,
            answer_confidence=confidence,
            confidence_score=uncertainty.fused_score,
            conflict=conflict,
            retrieval_rounds=total_rounds,
            clarification_needed=(record.decision == RetrievalDecision.CLARIFY),
            clarification_questions=[],
            sources=self._unique_sources(all_docs),
            elapsed_sec=round(time.time() - t0, 2),
            notes=[
                f"Decision: {record.decision.value}",
                f"Depth: {depth_str}",
                f"Uncertainty: {uncertainty.fused_score:.3f} ({uncertainty.level.value})",
                f"Conflict: {conflict.conflict_type.value if conflict else 'none'}",
                f"Tags: {conf_tags}",
            ],
        )

        Log.answer_box(result)

        # Print annotated answer
        if annotated and annotated.strip() != answer.strip():
            Log.section("CONFIDENCE-ANNOTATED ANSWER", C.GREEN)
            import textwrap
            for line in textwrap.wrap(annotated, W - 4):
                print(f"  {line}")

        return result


In [22]:


# ══════════════════════════════════════════════════════════════════════
# DEMO QUERIES — one per adaptive decision pathway
# ══════════════════════════════════════════════════════════════════════

DEMO_QUERIES = [
    # 1. SKIP — model is very confident, retrieval unnecessary
    (
        "What is the capital of France?",
        "Expected: SKIP — extremely high confidence, no retrieval needed",
    ),
    # 2. SKIP — opinion/reasoning, model can answer directly
    (
        "Is 17 a prime number?",
        "Expected: SKIP — deterministic reasoning, no corpus needed",
    ),
    # 3. SHALLOW — single-hop factual query about known concepts
    (
        "What does FLARE stand for and what problem does it solve?",
        "Expected: SHALLOW retrieval — specific technique, moderate confidence",
    ),
    # 4. STANDARD — moderate complexity, multiple aspects
    (
        "How does Self-RAG use reflection tokens to decide when to retrieve, "
        "and what benchmark results does it achieve?",
        "Expected: STANDARD retrieval — multi-aspect factual query",
    ),
    # 5. DEEP / MULTI_HOP — complex multi-step reasoning
    (
        "How does uncertainty estimation in Adaptive-RAG compare to the "
        "approach in Self-RAG, and what are the tradeoffs in calibration "
        "accuracy versus inference latency for each approach?",
        "Expected: DEEP or MULTI_HOP — requires bridging multiple papers",
    ),
    # 6. CLARIFY — genuinely ambiguous query
    (
        "What's the best way to handle it when things go wrong?",
        "Expected: CLARIFY — highly ambiguous, multiple valid interpretations",
    ),
]



In [48]:

# ══════════════════════════════════════════════════════════════════════
# SYSTEM BUILDER
# ══════════════════════════════════════════════════════════════════════

def build_system(cfg: Config) -> AdaptiveRAGEngine:
    Log.banner(
        "ADAPTIVE RAG — Uncertainty-Aware Retrieval",
        "Three-Signal Uncertainty · Adaptive Depth · Conflict Detection · Claim Annotation",
    )

    # ── Embeddings ─────────────────────────────────────────────────────
    em = EmbeddingManager.get(cfg)
    em.init()

    # ── Vector index ───────────────────────────────────────────────────
    index = AdaptiveVectorIndex(cfg, em)
    index.init_empty()

    # ── LLM ────────────────────────────────────────────────────────────
    llm = None
    if cfg.is_configured():
        Log.step("Connecting to Azure OpenAI")
        llm = AzureChatOpenAI(
            azure_endpoint=getattr(cfg, 'AI_FOUNDRY_PROJECT_ENDPOINT', 'https://idkopenai.services.ai.azure.com'),
            azure_deployment=getattr(cfg, 'AI_FOUNDRY_DEPLOYMENT_NAME', 'gpt-4o'),
            api_version=getattr(cfg, 'AI_FOUNDRY_API_VERSION', '2024-12-01-preview'),
            api_key=getattr(cfg, 'AI_FOUNDRY_API_KEY', ''),
            temperature=getattr(cfg, 'LLM_TEMPERATURE', 0.0),
            max_tokens=getattr(cfg, 'LLM_MAX_TOKENS', 512),
        )
        Log.ok(f"LLM ready — {cfg.AI_FOUNDRY_DEPLOYMENT_NAME}")
    else:
        Log.warn(
            "Azure OpenAI not configured — uncertainty will use heuristic fallback. "
            "Set AI_FOUNDRY_* values for full functionality."
        )

    # ── Ingest documents ───────────────────────────────────────────────
    if index.count() == 0:
        Log.step("Ingesting corpus…")

        # Attempt primary PDF (Adaptive-RAG paper)
        _try_pdf(cfg, index)

        Log.step(f"Ingesting {len(DEMO_DOCUMENTS)} demo documents")
        index.ingest(DEMO_DOCUMENTS)

        Log.step(f"Ingesting {len(KB_DOCS)} reference papers")
        index.ingest(KB_DOCS)
    else:
        Log.ok(f"Existing index: {index.count():,} chunks loaded from disk")

    # ── Reference table ────────────────────────────────────────────────
    Log.step("Reference papers:")
    for i, (title, url) in enumerate(ALL_REFERENCES, 1):
        Log.info(f"  [{i:2d}] {title}")
        Log.info(f"        {url}")

    # ── Engine ─────────────────────────────────────────────────────────
    engine = AdaptiveRAGEngine(cfg)
    engine.setup(index, em, llm)
    return engine



In [41]:

def _try_pdf(cfg: Config, index: AdaptiveVectorIndex):
    """Attempt to download and ingest the primary reference PDF."""
    try:
        import urllib.request
        from langchain_community.document_loaders import PyPDFLoader
        from corpus import PRIMARY_PDF

        pdf_path = PRIMARY_PDF.get("local", "./adaptive_rag_paper.pdf")
        url      = PRIMARY_PDF.get("url", "")
        source   = PRIMARY_PDF.get("source", "Adaptive-RAG Paper")

        if not Path(pdf_path).exists() and url:
            Log.info(f"Downloading PDF: {url}")
            urllib.request.urlretrieve(url, pdf_path)
            Log.ok(f"PDF saved → {pdf_path}")

        if Path(pdf_path).exists():
            pages     = PyPDFLoader(pdf_path).load()
            full_text = " ".join(p.page_content for p in pages if p.page_content.strip())
            index.ingest([{"id": "adaptive_rag_pdf", "source": source,
                           "url": url, "content": full_text}])
            Log.ok(f"PDF ingested: {len(pages)} pages")
    except Exception as exc:
        Log.warn(f"PDF unavailable ({exc}), skipping")


In [16]:


# ══════════════════════════════════════════════════════════════════════
# RUN MODES
# ══════════════════════════════════════════════════════════════════════

def run_demo(engine: AdaptiveRAGEngine, n: int = 0):
    """Run all demos (n=0) or a specific one (n=1–N)."""
    queries = DEMO_QUERIES if n == 0 else [DEMO_QUERIES[n - 1]]
    start_i = 1 if n == 0 else n
    print(f"\n{C.BOLD}{C.CYAN}Running {len(queries)} adaptive demo quer"
          f"{'y' if len(queries) == 1 else 'ies'}…{C.RESET}")

    for i, (query, note) in enumerate(queries, start_i):
        print(f"\n{C.BOLD}{C.YELLOW}{'▓'*W}")
        print(f"  Demo {i}: {note}")
        print(f"{'▓'*W}{C.RESET}")
        try:
            engine.query(query)
        except Exception as exc:
            Log.err(f"Demo {i} failed: {exc}")
        time.sleep(0.3)



In [17]:


def run_interactive(engine: AdaptiveRAGEngine):
    print(f"\n{C.BOLD}{C.CYAN}")
    print("╔══════════════════════════════════════════════════════════════╗")
    print("║   ADAPTIVE RAG — Uncertainty-Aware Retrieval                 ║")
    print("║   Interactive CLI                                            ║")
    print("║                                                              ║")
    print("║   Commands:                                                  ║")
    print("║     'demo N'  — run demo query N (1–6)                       ║")
    print("║     'demos'   — list all demo queries                        ║")
    print("║     'stats'   — print index statistics                       ║")
    print("║     'refs'    — list reference papers                        ║")
    print("║     'q'       — quit                                         ║")
    print("╚══════════════════════════════════════════════════════════════╝")
    print(C.RESET)

    while True:
        try:
            user = input(f"\n{C.BOLD}[adaptive-rag] Query:{C.RESET} ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nGoodbye!")
            break

        if not user:
            continue
        cmd = user.lower()

        if cmd in ("q", "quit", "exit"):
            break

        if cmd == "stats":
            Log.section("INDEX STATISTICS")
            Log.info(f"Chunks indexed: {engine.index.count():,}")
            continue

        if cmd == "refs":
            for i, (t, u) in enumerate(ALL_REFERENCES, 1):
                print(f"  [{i:2d}] {t}")
                print(f"        {u}")
            continue

        if cmd == "demos":
            for i, (q, note) in enumerate(DEMO_QUERIES, 1):
                print(f"  {i}. {note}")
                print(f"     {q[:70]}")
            continue

        if cmd.startswith("demo"):
            parts = cmd.split()
            if len(parts) > 1 and parts[1].isdigit():
                run_demo(engine, int(parts[1]))
            else:
                run_demo(engine)
            continue

        try:
            engine.query(user)
        except Exception as exc:
            Log.err(f"Error: {exc}")



In [4]:
# Extra demo questions for Adaptive RAG
extra_demo_questions = [
    "What is the exact role of label smoothing in the original Transformer training setup?",
    "How does BERT's masked language modeling differ from autoregressive next-token prediction?",
    "If evidence from retrieved sources conflicts, how should confidence be calibrated in the final answer?",
]

RUN_EXTRA_DEMOS = False
if RUN_EXTRA_DEMOS:
    cfg = Config()
    engine = build_system(cfg)
    for i, question in enumerate(extra_demo_questions, 1):
        print(f"\n[Extra Adaptive Demo {i}] {question}")
        engine.query(question)
else:
    print("Set RUN_EXTRA_DEMOS = True to execute extra Adaptive RAG examples.")

Set RUN_EXTRA_DEMOS = True to execute extra Adaptive RAG examples.


In [51]:
# Verification run (Azure OpenAI)
_cfg = Config()
_engine = build_system(_cfg)
for i, (_q, _note) in enumerate(DEMO_QUERIES[:2], 1):
    print(f"\n[Adaptive Verification {i}] {_note}")
    _engine.query(_q)


════════════════════════════════════════════════════════════════════════════
  ADAPTIVE RAG — Uncertainty-Aware Retrieval
  Three-Signal Uncertainty · Adaptive Depth · Conflict Detection · Claim Annotation
════════════════════════════════════════════════════════════════════════════
  ✓ AdaptiveVectorIndex: 39 chunks ready

▶ Connecting to Azure OpenAI
  ✓ LLM ready — gpt-4o
  ✓ Existing index: 39 chunks loaded from disk

▶ Reference papers:
  ·   [ 1] Adaptive-RAG (Jeong et al., 2024)
  ·         https://arxiv.org/abs/2403.14403
  ·   [ 2] Self-RAG (Asai et al., 2023)
  ·         https://arxiv.org/abs/2310.11511
  ·   [ 3] FLARE (Jiang et al., 2023)
  ·         https://arxiv.org/abs/2305.06983
  ·   [ 4] LLM Calibration (Kadavath et al., 2022)
  ·         https://arxiv.org/abs/2207.05221
  ·   [ 5] Semantic Uncertainty (Kuhn et al., 2023)
  ·         https://arxiv.org/abs/2302.09664
  ·   [ 6] Selective Prediction (Varshney et al., 2022)
  ·         https://arxiv.org/abs/2205.09598
  

In [31]:
from pathlib import Path
print('Path import OK:', Path is not None)

Path import OK: True
